# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mah-gie/Flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Title: Prioritizing SEO Metadata Interventions via Click Deficit Modeling**

**Abstract:**

Maximizing organic search traffic requires identifying which highly visible pages are failing to convert impressions into clicks. This study analyzes 3.6 million rows of daily Google Search Console performance data from March 2026 to evaluate true click-through rates. We engineered a Click Deficit Model that calculates expected clicks based on global median CTRs, replacing rigid binary rules with a dynamic baseline. The model successfully identified severe traffic bleeders, such as a single page with over 212,000 impressions missing 160 expected clicks, that naive thresholds completely ignored. Ultimately, the resulting ranked triage queue provides editorial teams with data-backed targets for immediate metadata interventions to recover lost search engagement.

## 1. Question

**Research Question:**
Which visible pages are under-capturing clicks relative to their search impression volume, and what specific metadata interventions should be prioritized to recover that traffic?

**Decision Supported:**
This model provides SEO and editorial teams with a prioritized, data-backed triage queue. It removes the guesswork from content updates by directing limited resources strictly to pages with proven visibility but failing engagement signals.

In [1]:
# Capstone Configuration
chosen_lane = "CTR / Engagement Opportunity Scoring"
primary_target = "gsc_clicks"
baseline_metric = "gsc_avg_position"

print(f"Capstone Lane Locked: {chosen_lane}")
print("Ready to pull the warehouse data.")

Capstone Lane Locked: CTR / Engagement Opportunity Scoring
Ready to pull the warehouse data.


## 2. Data
**Data Selection & Boundaries**

**Source Table:** `fact_content_daily_performance` from the FlyRank Hugging Face warehouse.

**Time Window:** March 2026. This provides a complete, 31-day recent snapshot to evaluate immediate click-through-rate (CTR) performance.

**Exclusions:** We strictly enforce `gsc_data_available = TRUE`. Pages without Google Search Console telemetry cannot be evaluated for search engagement. We also deliberately exclude all GA4 (Google Analytics) traffic metrics like social or direct sessions to prevent data leakage, ensuring the model focuses purely on search discoverability.

In [2]:
import duckdb
from google.colab import userdata
import pandas as pd

# 1. Authenticate and open the DuckDB connection
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Reaching into the Hugging Face warehouse...")

# 2. Pull the specific CTR data for March 2026
query = """
SELECT content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND gsc_data_available = TRUE
"""
df_capstone = con.sql(query).df()

print(f"Data successfully loaded! Total rows to analyze: {len(df_capstone)}")
print("\nQuick look at the raw data:")
print(df_capstone.head())

Reaching into the Hugging Face warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data successfully loaded! Total rows to analyze: 3611061

Quick look at the raw data:
            content_hash_id report_date  gsc_impressions  gsc_clicks  \
0  content_b7e512995f79d5a6  2026-03-01               20           0   
1  content_05597932fe4da067  2026-03-01                1           0   
2  content_7a105f548d9c6916  2026-03-01              125           1   
3  content_905aa32a0230694e  2026-03-01                7           0   
4  content_a3ea9792f793ec72  2026-03-01               11           0   

   gsc_avg_position  
0          3.350000  
1          0.000000  
2          4.928000  
3          4.000000  
4          2.272727  


## 3. Methodology

**Methodology & Assumptions**

**Assumptions:** As discovered in ML-07, the `gsc_avg_position` metric contains unnatural values (e.g., 0.0000). Therefore, we will weight `gsc_impressions` heavily as the true proxy for visibility.

**Features:** `total_impressions`, `avg_position`.

**Target Label:** `actual_ctr` (Calculated as clicks / impressions).

**Validation Design:** We will calculate an "Expected Clicks" baseline using the median CTR for a page's impression bucket, then rank pages by their "Click Deficit" (Expected Clicks - Actual Clicks).

Leakage Check: All metrics are derived from the exact same time window. No future engagement metrics or Google Analytics labels are used, ensuring the triage queue strictly scores search discoverability.

In [3]:
# 1. Aggregate the noisy daily rows into monthly page-level metrics
df_agg = df_capstone.groupby('content_hash_id').agg(
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean')
).reset_index()

# 2. Engineer our target feature: actual CTR
# Adding a tiny fraction (0.0001) prevents division by zero errors
df_agg['actual_ctr'] = df_agg['total_clicks'] / (df_agg['total_impressions'] + 0.0001)

# 3. Filter out pages that are completely dead (too low volume to matter)
df_model = df_agg[df_agg['total_impressions'] > 50].copy()

print(f"Aggregated down to {len(df_model)} unique, active pages ready for modeling.")
print("Preview of engineered features:")
print(df_model.head())

Aggregated down to 115723 unique, active pages ready for modeling.
Preview of engineered features:
            content_hash_id  total_impressions  total_clicks  avg_position  \
0  content_000005d4ced12088                 86             0     72.854861   
4  content_00014efc121d911d                116             1      4.964683   
5  content_000184dde41afe75               4885            15      3.576965   
7  content_0002bd310bf01f15                187             0     61.952029   
8  content_00032be2df0005ca                902             4      9.649411   

   actual_ctr  
0    0.000000  
4    0.008621  
5    0.003071  
7    0.000000  
8    0.004435  


## 4. Results (vs baseline)

**Results vs. Baseline**

**The Baseline Model:** Flagging pages with high impressions (>1,000) but exactly 0 clicks. This is too rigid and misses pages bleeding thousands of clicks just because they managed to get 1 or 2 accidental taps.

**The Deficit Model (Capstone):** We calculate the global median CTR across all active pages. We multiply this baseline rate by a page's total impressions to find its "Expected Clicks". By subtracting actual clicks, we find the "Click Deficit."

**The Verdict:** Instead of a dumb binary flag, we now have a mathematically ranked queue showing exactly how many clicks a page is underperforming by, sorted from highest traffic-bleed to lowest.

In [4]:
# 1. The Naive Baseline (ML-07 style)
df_model['baseline_flag'] = ((df_model['total_impressions'] > 1000) & (df_model['total_clicks'] == 0)).astype(int)

# 2. The Deficit Model (Our new intelligence)
global_median_ctr = df_model['actual_ctr'].median()
df_model['expected_clicks'] = df_model['total_impressions'] * global_median_ctr
df_model['click_deficit'] = df_model['expected_clicks'] - df_model['total_clicks']

# 3. Sort by the worst bleeders (highest deficit first)
df_model_ranked = df_model.sort_values(by='click_deficit', ascending=False)

print(f"Global Median CTR: {global_median_ctr:.4f}")
print(f"Pages flagged by Naive Baseline: {df_model['baseline_flag'].sum()}")
print("\nTop 5 Pages Bleeding the Most Clicks (Deficit Model):")
print(df_model_ranked[['content_hash_id', 'total_impressions', 'total_clicks', 'click_deficit']].head())

Global Median CTR: 0.0009
Pages flagged by Naive Baseline: 4378

Top 5 Pages Bleeding the Most Clicks (Deficit Model):
                 content_hash_id  total_impressions  total_clicks  \
47408   content_44f34c0a90047651             212404            24   
98003   content_8e1334d6356668e3             134984             1   
175924  content_fec55986a1868d62             124075             1   
131165  content_bdf60c86117079be             112429            12   
58911   content_559cdd76da9306de              97378             2   

        click_deficit  
47408      160.592119  
98003      116.309385  
175924     106.828794  
131165      85.707705  
58911       82.627462  


## 5. Limitations

**Limitations**

**Lack of Causality:** This model identifies exactly where the traffic bleed is happening, but cannot determine why. A high click deficit might be due to a terrible meta title, but it could also be caused by Google surfacing a rich snippet or AI overview that satisfies the user's question without them needing to click.

**Temporal Blindspots:** By aggregating into a monthly view to smooth out the noise, we lose visibility into sudden, mid-month algorithm drops.

In [5]:
print("Limitations documented conceptually in the Markdown cell above.")

Limitations documented conceptually in the Markdown cell above.


## 6. Ranked recommendations

**Ranked Recommendations Playbook**

**Priority 1 (Top 100 Deficit Pages):** Route immediately to the editorial team for Title and Meta Description rewrites. The high impressions prove the search demand exists; the external copy is simply failing to convert the click.

**Priority 2 (Deficit > 50):** Investigate search intent mismatch. Review the queries driving these impressions to ensure the page content actually answers what the user is looking for.

In [6]:
print("Ranked recommendations playbook detailed in the Markdown cell above.")

Ranked recommendations playbook detailed in the Markdown cell above.


## 7. Artifacts the paper embeds

**Artifacts Generated**

**The Master Triage Queue:** A complete `capstone_ranked_queue.csv` exported to the outputs folder, containing all active pages scored by their click deficit.

**Top 5 Bleeders Table:** The data table generated in Section 4 showing the most urgent metadata fixes (e.g., the page missing 160 expected clicks). This will be embedded directly into the final deployed research paper to prove the model's business value.

In [7]:
import os

# Create the outputs folder if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Save the final ranked model queue as our artifact
csv_path = 'work/outputs/capstone_ranked_queue.csv'
df_model_ranked.to_csv(csv_path, index=False)

print(f"Capstone artifact successfully saved to {csv_path}")
print("Self-check cleared. Paper is ready for deployment!")

Capstone artifact successfully saved to work/outputs/capstone_ranked_queue.csv
Self-check cleared. Paper is ready for deployment!


**ML-12 Deliverables**

**5-Minute Demo Outline**

**The Problem (1 min):** Highlighting how traditional SEO triage relies on guesswork or rigid, binary rules that miss nuance.

**The Solution (1 min):** Introducing the Click Deficit Model—calculating expected clicks based on global median CTRs and subtracting actual clicks.

**The Proof (2 min):** Showcasing our top bleeder (212,000+ impressions, only 24 clicks, missing 160 expected clicks) and comparing it to the naive baseline that missed it completely.

**The Action (1 min):** Handing off the ranked triage queue to the editorial team for targeted meta-title rewrites.

**Social-Post Cut**

Stop guessing which pages to fix first. For my FlyRank AI Capstone, I built a Click Deficit Model that analyzes millions of search telemetry rows to identify high-visibility pages bleeding the most traffic. Instead of a binary "good/bad" flag, it ranks URLs by exactly how many clicks they are under-capturing compared to a dynamic median CTR. Data-driven triage > guesswork.

**3-Sentence Employer-Facing Summary**

Engineered a Click Deficit Model using DuckDB and Python to analyze 3.6 million rows of search performance data. The model replaced a rigid rule-based system with a dynamic baseline to identify and rank high-impression web pages underperforming their expected click-through rates. Delivered a prioritized triage queue to guide targeted metadata interventions, optimizing editorial resources for maximum traffic recovery.

**Acknowledgments & Data Credit**

This capstone was built on the FlyRank ML Internship dataset. All real-world search telemetry and data infrastructure were provided by [FlyRank](https://flyrank.ai).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
